In [ ]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss

In [ ]:
print("Loading testing vectors...")
x_te = joblib.load('/kaggle/input/datasets/shrijitmittra/training-and-testing-datasets/x_testing_vector.pkl')
y_te = joblib.load('/kaggle/input/datasets/shrijitmittra/training-and-testing-datasets/y_testing_vector.pkl')

In [ ]:
print("Loading saved model and selector...")
sel = joblib.load('/kaggle/input/datasets/shrijitmittra/training-and-testing-datasets/sel.pkl')
ensemble = joblib.load('/kaggle/input/datasets/shrijitmittra/training-and-testing-datasets/model.pkl')

In [ ]:
print("Applying feature selection to test data...")
x_te_s = sel.transform(x_te)

In [ ]:
def evaluate_model(name, model, x_eval, y_eval):
    y_pred_local = model.predict(x_eval)
    y_prob_local = model.predict_proba(x_eval)

    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_eval, y_pred_local), 4),
        'Precision': round(precision_score(y_eval, y_pred_local), 4),
        'Recall': round(recall_score(y_eval, y_pred_local), 4),
        'F1': round(f1_score(y_eval, y_pred_local), 4),
        'Loss (Log Loss)': round(log_loss(y_eval, y_prob_local), 4),
    }

In [ ]:
evaluation_rows = []
for model_name, model in best_models.items():
    evaluation_rows.append(evaluate_model(model_name, model, x_te_s, y_te))
evaluation_rows.append(evaluate_model('Voting Ensemble', ensemble, x_te_s, y_te))

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(by='F1', ascending=False)
print('Evaluation Matrix:')
print(evaluation_df.to_string(index=False))

In [ ]:
y_pred = ensemble.predict(x_te_s)
y_prob = ensemble.predict_proba(x_te_s)

print('Accuracy:', round(accuracy_score(y_te, y_pred) * 100, 2), '%')
print('Log Loss:', round(log_loss(y_te, y_prob), 4))
print('\nClassification Report:\n', classification_report(y_te, y_pred, target_names=['Ham', 'Spam']))
print('Confusion Matrix:\n', confusion_matrix(y_te, y_pred))

In [ ]:
joblib.dump(ensemble, '/kaggle/working/model.pkl')
joblib.dump(sel, '/kaggle/working/sel.pkl')
print(f'Saved model.pkl and sel.pkl in: {BASE_DIR.resolve()}')